# Document Type Classification — **Max Accuracy** Pipeline (v10)

"
"목표: Public LB를 최대한 올리기 위한 실전 파이프라인입니다.

"
"핵심 업그레이드
"
"1) **StratifiedGroupKFold**(그룹=유사중복 클러스터)로 CV 누수/과대평가 감소 [Source](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)
"
"2) 문서 도메인에서 위험한 **HFlip/TTA 기본 OFF** (필요 시 scale TTA만)
"
"3) 불균형은 **Focal / Class-Balanced(Effective Number)** 스위치로 비교 [Source](https://openaccess.thecvf.com/content_CVPR_2019/html/Cui_Class-Balanced_Loss_Based_on_Effective_Number_of_Samples_CVPR_2019_paper.html)
"
"4) 앙상블은 기본 **logit mean**(softmax 전에 평균)
"
"5) (선택) **OCR+텍스트 임베딩** 멀티모달: 문서 타입은 키워드가 강한 단서라 성능을 자주 끌어올립니다(다만 시간/자원 비용 큼).

"
"주의: OCR은 느리므로 결과/임베딩을 **캐시**하도록 구성했습니다.


In [ ]:
# =====================
# Config
# =====================
CFG = {
    'seed': 42,
    'n_folds': 5,
    'folds_to_train': [0,1,2,3,4],

    # Paths
    'train_csv': '../data/train.csv',
    'sample_submission_csv': '../data/sample_submission.csv',
    'train_dir': '../data/train',
    'test_dir': '../data/test',

    # If you keep images in zips, fill these (optional)
    'train_zips': [],
    'test_zips': [],

    # GroupKFold grouping
    'group_th': 6,
    'group_prefix_bits': 18,

    # Model (자원 허용 시 backbone 다양화 앙상블 추천)
    # 예: ['convnext_base', 'swin_base_patch4_window7_224', 'efficientnet_b3']
    'model_names': ['efficientnet_b3'],
    'img_size': 384,

    # Train
    'batch_size': 24,
    'num_workers': 8,
    'epochs': 20,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'warmup_ratio': 0.10,

    # Loss preset: 'ce' | 'focal' | 'cb_ce' | 'cb_focal'
    'loss_name': 'cb_ce',
    'focal_gamma': 2.0,
    'cb_beta': 0.9999,
    # 문서 불균형에서 label smoothing은 신중(가중치와 동시 사용 지양)
    'label_smoothing': 0.0,

    # Optional sampler
    'use_weighted_sampler': False,

    # AMP
    'use_amp': True,
    'max_grad_norm': 1.0,

    # Early stopping
    'patience': 5,
    'min_delta': 1e-4,

    # TTA (문서는 flip 비추)
    'tta_enabled': False,
    'tta_scales': [1.0, 1.1],

    # Multimodal OCR
    'use_ocr_text': True,
    'ocr_langs': ['en'],
    'text_embed_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'text_embed_dim': 384,

    # Output
    'out_dir': 'outputs_v10_maxacc',
    'cache_dir': 'cache_v10',
    'save_oof_logits': True,
    'ensemble': 'logit_mean',
}
print(CFG)

In [ ]:
# =====================
# Imports & seed
# =====================
import os, io, time, random, math, zipfile
from collections import defaultdict

import numpy as np
import pandas as pd

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.optim import AdamW

import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__, 'device:', DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(CFG['seed'])

os.makedirs(CFG['out_dir'], exist_ok=True)
os.makedirs(CFG['cache_dir'], exist_ok=True)

## 1) Load CSV

In [ ]:
train_df = pd.read_csv(CFG['train_csv']).rename(columns={'ID':'image_id'})
test_df  = pd.read_csv(CFG['sample_submission_csv']).rename(columns={'ID':'image_id'})
num_classes = train_df['target'].nunique()
print('train:', train_df.shape, 'test:', test_df.shape, 'num_classes:', num_classes)
display(train_df.head())
display(train_df['target'].value_counts().sort_index())

## 2) Image reader (folder/zip 지원)

In [ ]:
class ZipImageStore:
    def __init__(self, zip_paths):
        self.zip_paths = [p for p in zip_paths if p]
        self.zips = [zipfile.ZipFile(p) for p in self.zip_paths]
        self.index = {}
        for z in self.zips:
            for n in z.namelist():
                if n.lower().endswith(('.jpg','.jpeg','.png')):
                    bn = os.path.basename(n)
                    if bn not in self.index:
                        self.index[bn] = z

    def has(self, image_id):
        return image_id in self.index

    def read_bgr(self, image_id):
        z = self.index[image_id]
        b = z.read(image_id)
        arr = np.frombuffer(b, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        return img

train_zip_store = ZipImageStore(CFG.get('train_zips', []))
test_zip_store  = ZipImageStore(CFG.get('test_zips',  []))

def read_image_bgr(image_id, split='train'):
    if split == 'train':
        p = os.path.join(CFG['train_dir'], image_id)
        if os.path.exists(p):
            return cv2.imread(p, cv2.IMREAD_COLOR)
        if train_zip_store.has(image_id):
            return train_zip_store.read_bgr(image_id)
    else:
        p = os.path.join(CFG['test_dir'], image_id)
        if os.path.exists(p):
            return cv2.imread(p, cv2.IMREAD_COLOR)
        if test_zip_store.has(image_id):
            return test_zip_store.read_bgr(image_id)
    raise FileNotFoundError(f'Image not found: {image_id} (split={split})')

_tmp = read_image_bgr(train_df.loc[0,'image_id'], split='train')
print('sample shape:', _tmp.shape)

## 3) Build groups (near-duplicate clustering) → StratifiedGroupKFold
CV–LB 괴리의 1순위는 '유사문서가 train/val에 동시에 들어가는 누수'일 가능성이 큽니다.
그래서 그룹을 만들고 StratifiedGroupKFold로 분할합니다. [Source](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)

In [ ]:
def ahash_u64_from_bgr(img_bgr, hash_size=8):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (hash_size, hash_size), interpolation=cv2.INTER_AREA)
    arr = g.astype(np.float32)
    avg = arr.mean()
    bits = (arr > avg).astype(np.uint64).reshape(-1)
    h = np.uint64(0)
    for b in bits:
        h = (h << np.uint64(1)) | np.uint64(b)
    return h

def hamming_u64(a,b):
    return int((int(a) ^ int(b)).bit_count())

def build_groups_ahash(ids, split='train', th=6, prefix_bits=18, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        if set(['image_id','group']).issubset(df.columns) and len(df)==len(ids):
            return df['group'].to_numpy(np.int32)

    hashes = np.zeros(len(ids), dtype=np.uint64)
    for i, image_id in enumerate(ids):
        img = read_image_bgr(image_id, split=split)
        hashes[i] = ahash_u64_from_bgr(img)

    shift = 64 - prefix_bits
    buckets = defaultdict(list)
    for i,h in enumerate(hashes):
        buckets[int(h >> np.uint64(shift))].append(i)

    n = len(ids)
    parent = np.arange(n, dtype=np.int32)
    rank = np.zeros(n, dtype=np.int16)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a,b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    for idxs in buckets.values():
        m = len(idxs)
        if m < 2:
            continue
        for ii in range(m):
            i = idxs[ii]
            for jj in range(ii+1, m):
                j = idxs[jj]
                if hamming_u64(hashes[i], hashes[j]) <= th:
                    union(i,j)

    roots = np.array([find(i) for i in range(n)], dtype=np.int32)
    _, group = np.unique(roots, return_inverse=True)
    group = group.astype(np.int32)

    if cache_path:
        pd.DataFrame({'image_id': ids, 'group': group}).to_csv(cache_path, index=False)
    return group

train_ids = train_df['image_id'].tolist()
group_cache = os.path.join(CFG['cache_dir'], 'train_groups_ahash_th{}.csv'.format(CFG['group_th']))
train_df['group'] = build_groups_ahash(train_ids, split='train', th=CFG['group_th'], prefix_bits=CFG['group_prefix_bits'], cache_path=group_cache)

gsize = train_df.groupby('group').size()
dup_img_rate = float((train_df['group'].map(gsize) >= 2).mean())
print('dup_img_rate:', f'{dup_img_rate:.3%}', 'n_groups:', train_df['group'].nunique(), 'max_group:', int(gsize.max()))

sgkf = StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for fold, (_, va_idx) in enumerate(sgkf.split(train_df['image_id'], train_df['target'], groups=train_df['group'])):
    train_df.loc[va_idx, 'fold'] = fold

assert (train_df['fold'] >= 0).all()
print(train_df['fold'].value_counts().sort_index())
print('leaky_groups:', int((train_df.groupby('group')['fold'].nunique() > 1).sum()), '(should be 0)')
train_df[['image_id','target','group','fold']].to_csv(os.path.join(CFG['out_dir'],'folds_groupkfold.csv'), index=False)

## 4) (선택) OCR + 텍스트 임베딩 캐시
정확도 최상 목표면 이미지+텍스트 멀티모달이 도움이 되는 경우가 많습니다.
- OCR은 느리므로 train/test 전체에 대해 1회만 수행 후 캐시
- 임베딩은 sentence-transformers로 생성해 캐시

설치가 필요할 수 있습니다: easyocr, sentence-transformers

In [ ]:
USE_OCR = bool(CFG.get('use_ocr_text', False))
print('USE_OCR:', USE_OCR)

def normalize_ocr_text(s: str) -> str:
    s = s.lower()
    s = ''.join([ch if ch.isalnum() else ' ' for ch in s])
    s = ' '.join(s.split())
    return s

if USE_OCR:
    try:
        import easyocr
        from sentence_transformers import SentenceTransformer
    except Exception as e:
        raise RuntimeError(
            'OCR 사용을 위해 easyocr & sentence-transformers 설치가 필요합니다. '
            '설치 후 다시 실행하세요. 예: !pip install easyocr sentence-transformers'        ) from e

    reader = easyocr.Reader(CFG['ocr_langs'], gpu=(DEVICE.type=='cuda'))
    text_model = SentenceTransformer(CFG['text_embed_model'])

    def ocr_text_from_bgr(img_bgr):
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        res = reader.readtext(img, detail=0, paragraph=True)
        if isinstance(res, list):
            txt = ' '.join([r for r in res if isinstance(r, str)])
        else:
            txt = str(res)
        return normalize_ocr_text(txt)

    def build_text_embeddings(ids, split, cache_path):
        if os.path.exists(cache_path):
            arr = np.load(cache_path)
            if arr.shape[0] == len(ids):
                return arr

        texts = []
        for i, image_id in enumerate(ids):
            img = read_image_bgr(image_id, split=split)
            texts.append(ocr_text_from_bgr(img))
            if (i+1) % 200 == 0:
                print(f'OCR {split}: {i+1}/{len(ids)}')

        emb = text_model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        emb = emb.astype(np.float32)
        np.save(cache_path, emb)
        return emb

    train_text_cache = os.path.join(CFG['cache_dir'], 'train_text_emb.npy')
    test_text_cache  = os.path.join(CFG['cache_dir'],  'test_text_emb.npy')

    train_text_emb = build_text_embeddings(train_df['image_id'].tolist(), 'train', train_text_cache)
    test_text_emb  = build_text_embeddings(test_df['image_id'].tolist(),  'test',  test_text_cache)
    print('train_text_emb:', train_text_emb.shape, 'test_text_emb:', test_text_emb.shape)
else:
    train_text_emb = None
    test_text_emb = None

## 5) Transforms (문서 친화, flip 제외)

In [ ]:
def pad_to_square(img, value=255):
    h,w = img.shape[:2]
    if h==w:
        return img
    s = max(h,w)
    top = (s-h)//2
    bottom = s-h-top
    left = (s-w)//2
    right = s-w-left
    return cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[value,value,value])

class PadSquare(A.ImageOnlyTransform):
    def __init__(self, always_apply=True, p=1.0, value=255):
        super().__init__(always_apply=always_apply, p=p)
        self.value = value
    def apply(self, img, **params):
        return pad_to_square(img, value=self.value)

def build_train_tf(img_size):
    return A.Compose([
        PadSquare(value=255),
        A.Resize(img_size, img_size),
        A.OneOf([
            A.Affine(rotate=(-7,7), translate_percent=(0.0, 0.02), scale=(0.95, 1.05), p=1.0),
            A.Perspective(scale=(0.02, 0.07), p=1.0),
        ], p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(0.15, 0.15, p=1.0),
            A.HueSaturationValue(8, 10, 8, p=1.0),
        ], p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(5.0, 25.0), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.4), p=1.0),
        ], p=0.3),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3,5), p=1.0),
            A.MotionBlur(blur_limit=(3,5), p=1.0),
        ], p=0.2),
        A.ImageCompression(quality_lower=35, quality_upper=100, p=0.3),
        A.CoarseDropout(max_holes=8, max_height=img_size//10, max_width=img_size//10, p=0.2),
        A.Normalize(),
        ToTensorV2(),
    ])

def build_test_tf(img_size):
    return A.Compose([
        PadSquare(value=255),
        A.Resize(img_size, img_size),
        A.Normalize(),
        ToTensorV2(),
    ])

train_tf = build_train_tf(CFG['img_size'])
test_tf  = build_test_tf(CFG['img_size'])

## 6) Dataset (image + optional text embedding)

In [ ]:
class DocDataset(Dataset):
    def __init__(self, df, split='train', tfm=None, text_emb=None):
        self.df = df.reset_index(drop=True)
        self.split = split
        self.tfm = tfm
        self.text_emb = text_emb

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['image_id']
        img = read_image_bgr(image_id, split=self.split)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.tfm is not None:
            img = self.tfm(image=img)['image']

        if self.text_emb is not None:
            txt = torch.tensor(self.text_emb[idx], dtype=torch.float32)
        else:
            txt = torch.zeros(int(CFG['text_embed_dim']), dtype=torch.float32)

        if 'target' in row:
            y = int(row['target'])
            return img, txt, y, image_id
        return img, txt, image_id

def make_loader(df, split, tfm, shuffle, sampler=None, text_emb=None):
    ds = DocDataset(df, split=split, tfm=tfm, text_emb=text_emb)
    return DataLoader(
        ds,
        batch_size=CFG['batch_size'],
        shuffle=(shuffle and sampler is None),
        sampler=sampler,
        num_workers=CFG['num_workers'],
        pin_memory=True,
        drop_last=(split=='train'),
    )

## 7) Loss (CE / Focal / Class-Balanced)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1)
        p = torch.exp(logp)
        logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, target.unsqueeze(1)).squeeze(1)
        loss = -((1 - p_t) ** self.gamma) * logp_t
        if self.weight is not None:
            w = self.weight.gather(0, target)
            loss = loss * w
        return loss.mean() if self.reduction=='mean' else loss.sum()

def class_balanced_weights(counts, beta=0.9999):
    counts = torch.tensor(counts, dtype=torch.float32)
    effective = 1.0 - torch.pow(beta, counts)
    w = (1.0 - beta) / (effective + 1e-12)
    w = w / w.sum() * len(counts)
    return w

def build_criterion(loss_name, class_counts, device):
    ls = float(CFG.get('label_smoothing', 0.0))
    if loss_name == 'ce':
        return nn.CrossEntropyLoss(label_smoothing=ls)
    if loss_name == 'focal':
        return FocalLoss(gamma=float(CFG['focal_gamma']))
    if loss_name == 'cb_ce':
        w = class_balanced_weights(class_counts, beta=float(CFG['cb_beta'])).to(device)
        return nn.CrossEntropyLoss(weight=w, label_smoothing=ls)
    if loss_name == 'cb_focal':
        w = class_balanced_weights(class_counts, beta=float(CFG['cb_beta'])).to(device)
        return FocalLoss(gamma=float(CFG['focal_gamma']), weight=w)
    raise ValueError(loss_name)

## 8) Model: image backbone + (optional) text MLP fusion

In [ ]:
class MultiModalDocModel(nn.Module):
    def __init__(self, model_name, num_classes, use_text=True, text_dim=384):
        super().__init__()
        self.use_text = use_text
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool='avg')
        img_dim = self.backbone.num_features
        if self.use_text:
            self.text_proj = nn.Sequential(
                nn.Linear(text_dim, 512),
                nn.ReLU(inplace=True),
                nn.Dropout(0.2),
                nn.Linear(512, 256),
                nn.ReLU(inplace=True),
            )
            fused_dim = img_dim + 256
        else:
            self.text_proj = None
            fused_dim = img_dim
        self.head = nn.Linear(fused_dim, num_classes)

    def forward(self, x_img, x_txt=None):
        feat = self.backbone(x_img)
        if self.use_text and x_txt is not None:
            t = self.text_proj(x_txt)
            feat = torch.cat([feat, t], dim=1)
        return self.head(feat)

def build_model(model_name, num_classes):
    return MultiModalDocModel(
        model_name=model_name,
        num_classes=num_classes,
        use_text=bool(CFG.get('use_ocr_text', False)),
        text_dim=int(CFG['text_embed_dim']),
    )

## 9) Warmup + Cosine scheduler (step-based)

In [ ]:
class WarmupCosine:
    def __init__(self, optimizer, total_steps, warmup_steps, min_lr=1e-6):
        self.optimizer = optimizer
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.min_lr = min_lr
        self.base_lrs = [g['lr'] for g in optimizer.param_groups]
        self.step_num = 0

    def step(self):
        self.step_num += 1
        for i, g in enumerate(self.optimizer.param_groups):
            base = self.base_lrs[i]
            if self.step_num <= self.warmup_steps and self.warmup_steps > 0:
                lr = base * self.step_num / self.warmup_steps
            else:
                t = (self.step_num - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
                lr = self.min_lr + 0.5*(base - self.min_lr)*(1 + math.cos(math.pi * t))
            g['lr'] = lr

    def get_lr(self):
        return [g['lr'] for g in self.optimizer.param_groups]

## 10) Train/Valid + EarlyStopping (macro F1)

In [ ]:
def macro_f1_from_logits(logits, targets):
    pred = logits.argmax(1)
    return f1_score(targets, pred, average='macro')

@torch.no_grad()
def valid_one_epoch(model, loader, criterion):
    model.eval()
    all_logits, all_targets = [], []
    losses = []
    for x, t, y, _ in loader:
        x = x.to(DEVICE)
        t = t.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(x, t)
        loss = criterion(logits, y)
        losses.append(loss.item())
        all_logits.append(logits.detach().cpu())
        all_targets.append(y.detach().cpu())
    all_logits = torch.cat(all_logits).numpy()
    all_targets = torch.cat(all_targets).numpy()
    f1 = macro_f1_from_logits(all_logits, all_targets)
    return float(np.mean(losses)), float(f1), all_logits, all_targets

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler=None):
    model.train()
    losses = []
    for x, t, y, _ in loader:
        x = x.to(DEVICE)
        t = t.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        if CFG['use_amp'] and DEVICE.type=='cuda':
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
                logits = model(x, t)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if CFG['max_grad_norm'] is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x, t)
            loss = criterion(logits, y)
            loss.backward()
            if CFG['max_grad_norm'] is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
            optimizer.step()
        scheduler.step()
        losses.append(loss.item())
    return float(np.mean(losses))

class EarlyStopper:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = -1e9
        self.count = 0
    def step(self, score):
        if score > self.best + self.min_delta:
            self.best = score
            self.count = 0
            return False
        self.count += 1
        return self.count >= self.patience

## 11) Train KFold (GroupKFold) — multi-backbone option
- model_names에 여러 backbone을 넣으면 아키텍처 앙상블까지 가능
- 가장 안정적인 앙상블은 **서로 다른 backbone + logit mean**

In [ ]:
best_paths = []
class_counts = train_df['target'].value_counts().sort_index().values

for model_name in CFG['model_names']:
    print('=== Training backbone:', model_name, '===')
    for fold in CFG['folds_to_train']:
        tr = train_df[train_df['fold'] != fold].copy()
        va = train_df[train_df['fold'] == fold].copy()

        tr_text = train_text_emb[tr.index.values] if USE_OCR else None
        va_text = train_text_emb[va.index.values] if USE_OCR else None

        sampler = None
        if CFG['use_weighted_sampler']:
            cnt = tr['target'].value_counts().to_dict()
            w = tr['target'].map(lambda t: 1.0 / cnt[int(t)]).values
            sampler = WeightedRandomSampler(weights=torch.tensor(w, dtype=torch.double), num_samples=len(w), replacement=True)

        tr_loader = make_loader(tr, 'train', train_tf, shuffle=True, sampler=sampler, text_emb=tr_text)
        va_loader = make_loader(va, 'train', test_tf, shuffle=False, sampler=None, text_emb=va_text)

        model = build_model(model_name, num_classes).to(DEVICE)
        optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
        total_steps = len(tr_loader) * CFG['epochs']
        warmup_steps = int(total_steps * CFG['warmup_ratio'])
        scheduler = WarmupCosine(optimizer, total_steps=total_steps, warmup_steps=warmup_steps, min_lr=1e-6)
        criterion = build_criterion(CFG['loss_name'], class_counts, DEVICE)
        scaler = torch.cuda.amp.GradScaler(enabled=(CFG['use_amp'] and DEVICE.type=='cuda'))
        es = EarlyStopper(patience=CFG['patience'], min_delta=CFG['min_delta'])

        best_f1 = -1
        best_path = os.path.join(CFG['out_dir'], f'best_{model_name}_fold{fold}.pth')

        for epoch in range(1, CFG['epochs']+1):
            t0 = time.time()
            tr_loss = train_one_epoch(model, tr_loader, criterion, optimizer, scheduler, scaler=scaler)
            va_loss, va_f1, _, _ = valid_one_epoch(model, va_loader, criterion)
            lr_now = scheduler.get_lr()[0]
            print(f'[M {model_name}][F {fold}][Ep {epoch:02d}] lr={lr_now:.2e} tr_loss={tr_loss:.4f} | va_loss={va_loss:.4f} va_f1={va_f1:.4f} | {time.time()-t0:.1f}s')
            if va_f1 > best_f1:
                best_f1 = va_f1
                torch.save({'model': model.state_dict(), 'cfg': CFG}, best_path)
            if es.step(va_f1):
                print(f'  EarlyStop. best_f1={best_f1:.4f}')
                break

        best_paths.append(best_path)

print('Saved checkpoints:', len(best_paths))
for p in best_paths[:5]:
    print(p)

## 12) Inference + Logit Mean Ensemble → submission.csv

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, tfm=None, text_emb=None):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.text_emb = text_emb
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        image_id = self.df.loc[idx,'image_id']
        img = read_image_bgr(image_id, split='test')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.tfm is not None:
            img = self.tfm(image=img)['image']
        if self.text_emb is not None:
            txt = torch.tensor(self.text_emb[idx], dtype=torch.float32)
        else:
            txt = torch.zeros(int(CFG['text_embed_dim']), dtype=torch.float32)
        return img, txt, image_id

def make_test_loader(df, tfm, text_emb=None):
    ds = TestDataset(df, tfm=tfm, text_emb=text_emb)
    return DataLoader(ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    outs = []
    ids = []
    for x, t, image_id in loader:
        x = x.to(DEVICE)
        t = t.to(DEVICE)
        logits = model(x, t)
        outs.append(logits.detach().cpu().numpy())
        ids += list(image_id)
    return np.concatenate(outs, axis=0), ids

test_loader = make_test_loader(test_df, test_tf, text_emb=(test_text_emb if USE_OCR else None))

all_logits = []
for p in best_paths:
    base = os.path.basename(p)
    # best_{model}_foldK.pth
    model_name = base.split('_fold')[0].replace('best_', '')
    model = build_model(model_name, num_classes).to(DEVICE)
    ckpt = torch.load(p, map_location='cpu')
    model.load_state_dict(ckpt['model'])
    logits, ids = predict_logits(model, test_loader)
    all_logits.append(logits)

all_logits = np.stack(all_logits, axis=0)
ens_logits = all_logits.mean(axis=0)
probs = torch.softmax(torch.tensor(ens_logits), dim=1).numpy()
pred = probs.argmax(1)
sub = pd.DataFrame({'ID': ids, 'target': pred})
sub_path = os.path.join(CFG['out_dir'], 'submission.csv')
sub.to_csv(sub_path, index=False)
print('saved:', sub_path)
display(sub.head())
np.save(os.path.join(CFG['out_dir'], 'test_probs.npy'), probs.astype(np.float16))